# Classificatore data-driven dei temi politici

## Scopo

Questo notebook costruirà una classificazione esplorativa dei temi emergenti nella stampa politica italiana. Il percorso previsto parte da microtemi estratti in modo data-driven e arriva a possibili macrotemi sottoposti a validazione umana.

L'unità di analisi è l'articolo. Un **microtema** descrive un argomento specifico ricorrente; un **macrotema** raggruppa microtemi affini; un **evento contingente** è un fatto circoscritto nel tempo e non va automaticamente interpretato come tema strutturale.

## Input atteso

Il notebook leggerà `data/raw/mediacloud_fulltext.jsonl`, prodotto dalla pipeline Media Cloud e harvester. I campi attesi sono `url`, `domain`, `seendate`, `title`, `text`, `chars` e, quando disponibile, `language`. URL, fonte e data saranno conservati negli output destinati a Tableau.

## Principi di interpretazione

La prima baseline utilizzerà TF-IDF + NMF. Il sentiment generale di un articolo non verrà usato come misura dello stance verso un partito e la semplice menzione di leader o partiti non comporterà un'attribuzione automatica.

Eseguire le celle nell'ordine indicato. Le etichette dei topic sono **provvisorie**: la loro interpretazione e l'eventuale aggregazione in macrotemi restano un passaggio umano, documentato nell'output di revisione.

## 1. Dipendenze e configurazione

Se necessario, dal root del repository eseguire `pip install -r requirements.txt`. Questa baseline richiede anche `pandas` e `scikit-learn`.

In [ ]:
from pathlib import Path
import re

import pandas as pd
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

# Il notebook vive in notebooks/: risaliamo in modo indipendente dalla cartella di avvio.
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

INPUT = ROOT / 'data' / 'raw' / 'mediacloud_fulltext.jsonl'
OUTPUT_DIR = ROOT / 'data' / 'processed'
N_TOPICS = 12          # ipotesi iniziale: confrontare 8, 12 e 16 in revisione
N_TERMS = 12           # parole caratteristiche mostrate per topic
MIN_DF = 3             # un termine deve comparire in almeno 3 articoli
MAX_DF = 0.85          # elimina formule ricorrenti in quasi tutto il corpus
RANDOM_STATE = 42


## 2. Caricamento e controlli

Il JSONL è prodotto da `src/mediacloud_fulltext.py`. Conserviamo i metadati originali per permettere il controllo manuale di ogni assegnazione.

In [ ]:
if not INPUT.exists():
    raise FileNotFoundError(
        f'Input non trovato: {INPUT}\n'
        'Prima eseguire: python src/mediacloud_spike.py e poi python src/mediacloud_fulltext.py'
    )

raw = pd.read_json(INPUT, lines=True)
required = {'url', 'domain', 'seendate', 'title', 'text'}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f'Campi mancanti nel JSONL: {sorted(missing)}')

print(f'Articoli letti: {len(raw):,}')
display(raw[['seendate', 'domain', 'title', 'language', 'chars']].head())
display(raw['language'].fillna('unknown').value_counts().rename_axis('language').to_frame('articoli'))


## 3. Preparazione del corpus

La baseline è limitata agli articoli identificati come italiani. Deduplichiamo per URL e rimuoviamo record senza testo sufficiente; questa scelta va riportata insieme ai conteggi finali.

In [ ]:
def normalizza_testo(value):
    value = '' if pd.isna(value) else str(value).lower()
    value = re.sub(r'https?://\S+|www\.\S+', ' ', value)
    value = re.sub(r'[^a-zàèéìòóù\s]', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()

corpus = raw.copy()
corpus = corpus.drop_duplicates(subset='url')
corpus = corpus[corpus['language'].fillna('it').eq('it')].copy()
corpus['title'] = corpus['title'].fillna('')
corpus['text'] = corpus['text'].fillna('')
corpus['testo_modello'] = (corpus['title'] + ' ' + corpus['text']).map(normalizza_testo)
corpus = corpus[corpus['testo_modello'].str.len() >= 300].copy()

if len(corpus) < 10:
    raise ValueError('Corpus insufficiente: servono almeno 10 articoli italiani con testo valido.')

print(f'Articoli nel corpus analizzato: {len(corpus):,}')
print(f'Intervallo date: {corpus.seendate.min()} → {corpus.seendate.max()}')


## 4. Topic modeling esplorativo (TF-IDF + NMF)

NMF assegna a ogni articolo un peso per ciascun topic. La lista di stopword sotto è deliberatamente piccola e trasparente: dopo la prima revisione può essere estesa solo con termini chiaramente non tematici (ad esempio formule editoriali), mai con parole politiche sostantive.

In [ ]:
STOPWORDS_IT = {
    'anche', 'ancora', 'avere', 'come', 'con', 'contro', 'dalla', 'dalle', 'dello',
    'della', 'delle', 'dopo', 'essere', 'fatto', 'fare', 'gli', 'ha', 'hanno',
    'legge', 'nelle', 'non', 'ogni', 'per', 'perché', 'più', 'quale', 'quello',
    'questa', 'questo', 'sarà', 'sono', 'sua', 'sue', 'sul', 'sulla', 'tutti',
    'una', 'uno', 'verso', 'oggi', 'ieri', 'dice', 'secondo'
}

# Se MIN_DF è troppo alto per un piccolo campione, abbassarlo mantenendo traccia della scelta.
min_df = min(MIN_DF, max(1, len(corpus) // 10))
vectorizer = TfidfVectorizer(
    stop_words=sorted(STOPWORDS_IT), ngram_range=(1, 2), min_df=min_df, max_df=MAX_DF,
    sublinear_tf=True, strip_accents='unicode'
)
X = vectorizer.fit_transform(corpus['testo_modello'])
if X.shape[1] < 2:
    raise ValueError('Vocabolario insufficiente: controllare testo, lingua e parametri TF-IDF.')

n_topics = min(N_TOPICS, len(corpus) - 1, X.shape[1] - 1)
if n_topics < 2:
    raise ValueError('Servono almeno due topic stimabili.')

nmf = NMF(n_components=n_topics, init='nndsvda', random_state=RANDOM_STATE, max_iter=500)
W = nmf.fit_transform(X)
H = nmf.components_
feature_names = vectorizer.get_feature_names_out()
print(f'Matrice TF-IDF: {X.shape[0]:,} articoli × {X.shape[1]:,} termini | topic: {n_topics}')


In [ ]:
def termini_topic(components, features, n_terms=N_TERMS):
    rows = []
    for topic_id, weights in enumerate(components):
        top = weights.argsort()[-n_terms:][::-1]
        rows.append({
            'topic_id': topic_id,
            'termini_caratteristici': ', '.join(features[i] for i in top),
        })
    return pd.DataFrame(rows)

topic_terms = termini_topic(H, feature_names)
display(topic_terms)


## 5. Revisione umana e export

`topic_id` non è una categoria sostantiva. Aprire `news_topic_review.csv`, leggere termini e articoli esemplari, poi compilare `macrotema_validato` in `news_topic_terms.csv`. Solo dopo questa validazione il mapping può alimentare la tabella gold.

In [ ]:
topic_columns = [f'topic_{i}_peso' for i in range(n_topics)]
pesi = pd.DataFrame(W, columns=topic_columns, index=corpus.index)
corpus['estratto'] = corpus['text'].str.replace(r'\s+', ' ', regex=True).str.slice(0, 500)
review = corpus[['url', 'domain', 'seendate', 'title', 'language', 'chars', 'estratto']].join(pesi)
review['topic_id'] = W.argmax(axis=1)
review['confidenza_topic'] = W.max(axis=1) / W.sum(axis=1).clip(min=1e-12)
review = review.merge(topic_terms, on='topic_id', how='left')
review = review.sort_values(['topic_id', 'confidenza_topic'], ascending=[True, False])

# Il mapping resta vuoto finché non viene verificato: evita di scambiare cluster per categorie finali.
topic_terms['macrotema_validato'] = ''
topic_terms['note_revisione'] = ''
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
review.to_csv(OUTPUT_DIR / 'news_topic_review.csv', index=False, encoding='utf-8-sig')
topic_terms.to_csv(OUTPUT_DIR / 'news_topic_terms.csv', index=False, encoding='utf-8-sig')

display(review[['topic_id', 'confidenza_topic', 'title', 'termini_caratteristici', 'estratto']].head(12))
print('Creati:')
print(f' - {OUTPUT_DIR / "news_topic_review.csv"}')
print(f' - {OUTPUT_DIR / "news_topic_terms.csv"}')
